# Probability

Solve unknown event probabilities and work with discrete distributions, Bayes updates, event probabilities, densities, finite-bag sampling, and grouped rates.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
import ProbabilityUtils
importlib.reload(ProbabilityUtils)
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

Python 3.11.17 | shared cache: D:\NUS\Y2S1\EE2211\solvers\ee2211-solver\notebooks\.cache\matrices.sqlite3


## Solve probabilities from known values
Enter any known values for events A, B, and C, then mark the quantities you want with
`???`. `P(...)` wrappers and lowercase names are both accepted.

| Input | Meaning |
| --- | --- |
| `a`, `b`, `c` or `P(C)` | Probability of an individual event |
| `a*b` | A and B: intersection, not multiplication of the two probabilities |
| `a+b` | A or B: union, including their overlap |
| `a\|b` | Probability of A given B |
| `a*b*c` / `a+b+c` | Triple intersection / union of three events |
| `a\|(b+c)` / `a\|(b*c)` | Conditional on a combined event |
| `~a` or `a'` | Complement: A does not occur |

One assignment per line, or separate assignments with semicolons. Values may be
decimals, fractions, or percentages. Parentheses combine events, for example
`~(a+b)` or `a|~b`. You can request more than one unknown.

The default example gives **P(B) = 2/5 = 0.4**. You could also enter only
`a*b = 0.1`, `a|b = 0.25`, and `b = ???` to obtain the same answer.
A third-event example is `a*c = 0.12`, `a|c = 0.3`, `c = ???`, which
returns **P(C) = 0.4**. Complements such as `~c` and `c'` also work.
The solver does not assume independence. Insufficient information produces bounds;
inconsistent values produce an input error. A conditional probability requires a
positive conditioning probability. Square brackets include endpoints; round
brackets exclude them. Use exact fractions for repeating decimals.

In [17]:
# * = intersection, + = union, | = conditional, ~ = complement; events A, B, and C.
statements = """
b = 0.1
a|b = 0.9
a|~b = 0.2
~a = ?
"""  # Known values: decimal, fraction or percentage. Mark each unknown with ???.
show_related = False  # True shows other event probabilities and any feasible ranges.

In [18]:
from ProbabilityUtils import solve_probabilities, format_probability_result

event_solution = solve_probabilities(statements, include_related=boolean(show_related))
display(pd.DataFrame([
    {"Probability": answer["expression"], "Answer": format_probability_result(answer)}
    for answer in event_solution["results"]
]))
for answer in event_solution["results"]:
    if answer["value"] is not None and not answer["undefined_possible"]:
        cache.save(answer["expression"], [[float(answer["value"])]], source="Solved probability")
if event_solution["related"]:
    display(Markdown("**Related probabilities**"))
    display(pd.DataFrame([
        {"Probability": answer["expression"], "Answer": format_probability_result(answer)}
        for answer in event_solution["related"]
    ]))

,Probability,Answer
0,P(~A),73/100 = 0.73


## 1. Discrete distribution
Each row contains outcome and probability. At most one probability may be `?`.
Probabilities are checked and must sum to 1. Comparisons are `>=`, `>`, `<=`,
`<`, `=`, or `between` (both bounds included). The CDF combines repeated outcomes.

In [4]:
pmf = '-2 1/4; 0 ?; 3 1/2'  # Rows: outcome probability; at most one probability may be ?.
comparison = '>='  # >= | > | <= | < | = | between.
boundary = 0  # Numeric boundary, or lower endpoint for between; fractions allowed.
upper = None  # Upper endpoint for between (inclusive); otherwise ignored.

In [5]:
from ProbabilityUtils import discrete_distribution

distribution_table = matrix(pmf, allow_missing=True)
if distribution_table.shape[1] != 2:
    raise ValueError("Enter outcome and probability in exactly two columns.")
distribution = discrete_distribution(distribution_table[:, 0], distribution_table[:, 1], comparison, number(boundary), number(upper) if comparison == "between" else None)
show_metrics({key: distribution[key] for key in ("event_probability", "mean", "variance", "std")})
display(pd.DataFrame({"Outcome": distribution["outcomes"], "Probability": distribution["probabilities"], "Inferred": distribution["unknown"]}))
display(pd.DataFrame({"x": distribution["sorted_outcomes"], "P(X <= x)": distribution["cdf"]}))
cache.save("distribution", np.column_stack((distribution["outcomes"], distribution["probabilities"])), source="Completed PMF")
cache.save("event_probability", [[distribution["event_probability"]]], source="Discrete event")

,Value
event_probability,0.75
mean,1.0
variance,4.5
std,2.1213203436


,Outcome,Probability,Inferred
0,-2.0,0.25,False
1,0.0,0.25,True
2,3.0,0.50,False


,x,P(X <= x)
0,-2.0,0.25
1,0.0,0.50
2,3.0,1.00


'0cff2e27e4584de4bf43f6aff59bab12'

## 2. Bayes update
Rows are mutually exclusive, exhaustive groups. Each row contains prior
P(group) and likelihood P(event | group). One prior may be `?`.
Set `event_occurred = False` to condition on the event not occurring.
Posterior = prior × observed likelihood / evidence probability.

In [6]:
priors = '0.30 0.10; 0.70 0.04'  # Rows: group prior, event likelihood; one prior may be ?.
event_occurred = True  # True conditions on event; False on its complement.

In [7]:
from ProbabilityUtils import bayes_update

bayes_table = matrix(priors, allow_missing=True)
if bayes_table.shape[1] != 2:
    raise ValueError("Enter prior and likelihood in exactly two columns.")
bayes = bayes_update(bayes_table[:, 0], bayes_table[:, 1], boolean(event_occurred))
print("Evidence probability:", bayes["evidence"])
display(pd.DataFrame({"Prior": bayes["priors"], "Observed likelihood": bayes["likelihoods"], "Joint": bayes["joint"], "Posterior": bayes["posterior"]}))
show("posterior", bayes["posterior"].reshape(-1, 1), cache=cache, source="Bayes update")

Evidence probability: 0.057999999999999996


,Prior,Observed likelihood,Joint,Posterior
0,0.3,0.10,0.030,0.5172413793
1,0.7,0.04,0.028,0.4827586207


**posterior**

,0
0,0.5172413793
1,0.4827586207


## 3. Two events
Supply P(A), P(B), and a known `intersection` or `union` probability. The joint
table has rows A/not A and columns B/not B. Conditional probabilities are undefined
when their conditioning event has probability zero.

In [8]:
p_a = 0.5  # P(A), from 0 to 1; fractions such as 1/2 accepted as text.
p_b = 0.4  # P(B), from 0 to 1.
known = 0.2  # Known joint or union probability, from 0 to 1.
known_kind = 'intersection'  # intersection | union.

In [9]:
from ProbabilityUtils import event_probabilities

events = event_probabilities(number(p_a), number(p_b), number(known), known_kind)
show_metrics({key: events[key] for key in ("intersection", "union", "p_a_given_b", "p_b_given_a", "independent", "mutually_exclusive")})
display(pd.DataFrame(events["joint_table"], index=["A", "not A"], columns=["B", "not B"]))
cache.save("joint_probability_table", events["joint_table"], source="Two events")

,Value
intersection,0.2
union,0.7
p_a_given_b,0.5
p_b_given_a,0.4
independent,True
mutually_exclusive,False


,B,not B
A,0.2,0.3
not A,0.2,0.3


'06bf9a80c1b94d4a9c175ef5bf6e2a0e'

## 4. Continuous density
The calculator verifies nonnegativity, continuity on finite support, and total
area 1, then integrates the query interval. The density is zero outside support.
An exact point has probability zero. Use `x` as the density variable.

In [10]:
pdf = '2*x'  # Density using x; * multiplies, ^ or ** raises powers.
support = (0.0, 1.0)  # (low, high), finite with low < high.
interval = (0.5, 1.0)  # (low, high) query bounds; may extend outside support.

In [11]:
from ProbabilityUtils import density_probability

density = density_probability(pdf, *(number(v) for v in support), *(number(v) for v in interval))
display(density["expression"])
show_metrics({key: density[key] for key in ("total_area", "probability")})
cache.save("interval_probability", [[density["probability"]]], source="Density integral")

2*x

,Value
total_area,1.0
probability,0.75


'99ef73d79c3549bbb3781dbbeeef5b2f'

## 5. Sampling without replacement
Draw uniformly from N items, including K marked items. X is the marked count in
n draws. The event comparisons are the same as in section 1. Counts are integers.
The result includes an exact fraction as well as its decimal value.

$$P(X=k)=\frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}.$$

In [12]:
N = 6  # Total items, integer from 1 to 10000.
K = 2  # Marked items, integer from 0 to N.
n = 2  # Draw count, integer from 0 to min(N, 1000).
event = '>='  # >= | > | <= | < | = | between.
count = 1  # Nonnegative integer boundary or lower count.
max_count = None  # Inclusive upper integer for between; otherwise ignored.

In [13]:
from ProbabilityUtils import sampling_without_replacement

sampling = sampling_without_replacement(integer(N), integer(K), integer(n),
    event, integer(count), integer(max_count) if event == "between" else None)
show_metrics({key: sampling[key] for key in ("exact_probability", "event_probability", "mean", "variance")})
display(pd.DataFrame({"Marked items drawn": sampling["outcomes"], "Probability": sampling["probabilities"]}))
cache.save("sampling_probability", [[sampling["event_probability"]]], source="Without replacement")

,Value
exact_probability,3/5
event_probability,0.6
mean,0.6666666667
variance,0.3555555556


,Marked items drawn,Probability
0,0.0,0.4000000000
1,1.0,0.5333333333
2,2.0,0.0666666667


'99424ff2fed24b4fb0cccc598b72e140'

## 6. Compare grouped rates
Each row is a group with four counts: **A successes, A total, B successes, B total**.
Pooled rates use total successes / total cases, with different weights for A and B.
The default example shows Simpson's reversal: A leads in each group but B leads
after pooling. Pooled rates are not the unweighted mean of group percentages.

In [14]:
groups = '9 10 80 100; 20 100 1 10'  # Rows: A successes, A total, B successes, B total; integer counts.

In [15]:
from ProbabilityUtils import compare_grouped_rates

rates = compare_grouped_rates(matrix(groups))
display(pd.DataFrame({"A rate": rates["rates"][:, 0], "B rate": rates["rates"][:, 1],
    "A weight": rates["weights"][:, 0], "B weight": rates["weights"][:, 1]}))
show_metrics({"Pooled A": rates["pooled"][0], "Pooled B": rates["pooled"][1], "Simpson reversal": rates["reversal"]})
show("grouped_rates", rates["rates"], cache=cache, source="Grouped success rates")
show("pooled_rates", rates["pooled"].reshape(1, -1), cache=cache, source="Pooled success rates")

,A rate,B rate,A weight,B weight
0,0.9,0.8,0.0909090909,0.9090909091
1,0.2,0.1,0.9090909091,0.0909090909


,Value
Pooled A,0.2636363636
Pooled B,0.7363636364
Simpson reversal,True


**grouped_rates**

,0,1
0,0.9,0.8
1,0.2,0.1


**pooled_rates**

,0,1
0,0.2636363636,0.7363636364
